### Task 3: Interactive Plotly Report

This notebook recreates three Matplotlib-style charts as interactive Plotly Express visuals and adds one extra interactive chart type.
Dataset: Titanic


In [5]:
import os
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Keep the default theme clean and readable for all figures.
pio.templates.default = "plotly_white"

base_folder = "c:/Users/PM/Desktop/Internship/AI-Data-Engineering-Internship/Week-6 May_26"
data_path = os.path.join(base_folder, "Titanic-Dataset.csv")
output_folder = os.path.join(base_folder, "Task3")
os.makedirs(output_folder, exist_ok=True)

data_path

'c:/Users/PM/Desktop/Internship/AI-Data-Engineering-Internship/Week-6 May_26\\Titanic-Dataset.csv'

In [6]:
# Load the Titanic dataset and create a working copy.
df = pd.read_csv(data_path)
plot_df = df.copy()

# Convert fields used in the charts to numeric values.
plot_df["Age"] = pd.to_numeric(plot_df["Age"], errors="coerce")
plot_df["Fare"] = pd.to_numeric(plot_df["Fare"], errors="coerce")

# Add readable labels for categorical storytelling.
plot_df["SurvivedLabel"] = plot_df["Survived"].map({0: "Did not survive", 1: "Survived"})
plot_df["ClassLabel"] = "Class " + plot_df["Pclass"].astype(str)

# Create a compact age band for grouped summaries.
age_bins = [0, 10, 20, 30, 40, 50, 60, 70, 80]
plot_df["AgeBand"] = pd.cut(plot_df["Age"], bins=age_bins, right=False)
plot_df["AgeBandLabel"] = plot_df["AgeBand"].astype(str)

plot_df[["Survived", "Pclass", "Sex", "Age", "Fare", "SurvivedLabel"]].head()

,Survived,Pclass,Sex,Age,Fare,SurvivedLabel
0,0,3,male,22.0,7.2500,Did not survive
1,1,1,female,38.0,71.2833,Survived
2,1,3,female,26.0,7.9250,Survived
3,1,1,female,35.0,53.1000,Survived
4,0,3,male,35.0,8.0500,Did not survive


### Chart 1: Grouped bar chart
This recreates the bar-chart style summary using interactive grouped bars. I color-encode survival status so the class comparison remains visible while also showing how the passenger mix changes within each class.

In [7]:
# Aggregate the data so the bar chart shows class counts split by survival status.
bar_df = (
    plot_df.groupby(["Pclass", "SurvivedLabel"], as_index=False)
    .agg(
        PassengerCount=("PassengerId", "count"),
        AvgAge=("Age", "mean"),
        AvgFare=("Fare", "mean"),
    )
)

bar_fig = px.bar(
    bar_df,
    x="Pclass",
    y="PassengerCount",
    color="SurvivedLabel",
    barmode="group",
    text="PassengerCount",
    hover_data={
        "AvgAge": ":.1f",
        "AvgFare": ":.2f",
        "PassengerCount": True,
    },
    color_discrete_map={"Did not survive": "#D95F02", "Survived": "#1B9E77"},
    labels={
        "Pclass": "Passenger class",
        "PassengerCount": "Passenger count",
        "SurvivedLabel": "Survival status",
    },
    title="Titanic Passenger Count by Class and Survival Status",
)

bar_fig.update_traces(textposition="outside")
bar_fig.show()

bar_html_path = os.path.join(output_folder, "task3_bar_chart.html")
bar_fig.write_html(bar_html_path)
bar_html_path

'c:/Users/PM/Desktop/Internship/AI-Data-Engineering-Internship/Week-6 May_26\\Task3\\task3_bar_chart.html'

### Chart 2: Scatter plot
This is the interactive version of the age-versus-fare chart. I use survival status for color so the two groups are easy to compare, and the hover panel exposes multiple passenger details instead of forcing labels onto the plot.

In [8]:
scatter_df = plot_df.dropna(subset=["Age", "Fare"]).copy()

scatter_fig = px.scatter(
    scatter_df,
    x="Age",
    y="Fare",
    color="SurvivedLabel",
    hover_data={
        "Sex": True,
        "Pclass": True,
        "SibSp": True,
        "Parch": True,
        "Age": ":.1f",
        "Fare": ":.2f",
    },
    color_discrete_map={"Did not survive": "#D95F02", "Survived": "#1B9E77"},
    labels={
        "Age": "Age (years)",
        "Fare": "Fare (pounds sterling)",
        "SurvivedLabel": "Survival status",
    },
    title="Titanic Age vs Fare by Survival Status",
)

scatter_fig.show()

scatter_html_path = os.path.join(output_folder, "task3_scatter_chart.html")
scatter_fig.write_html(scatter_html_path)
scatter_html_path

'c:/Users/PM/Desktop/Internship/AI-Data-Engineering-Internship/Week-6 May_26\\Task3\\task3_scatter_chart.html'

### Chart 3: Box plot
This box plot recreates the class-vs-fare comparison in an interactive form. I keep the class grouping, use survival status for color, and expose age, sex, and fare values on hover so the chart tells a fuller passenger story.

In [9]:
box_df = plot_df.dropna(subset=["Fare"]).copy()
box_df["ClassLabel"] = "Class " + box_df["Pclass"].astype(str)

box_fig = px.box(
    box_df,
    x="ClassLabel",
    y="Fare",
    color="SurvivedLabel",
    points="outliers",
    hover_data={
        "Sex": True,
        "Age": ":.1f",
        "Fare": ":.2f",
        "Embarked": True,
    },
    color_discrete_map={"Did not survive": "#D95F02", "Survived": "#1B9E77"},
    labels={
        "ClassLabel": "Passenger class",
        "Fare": "Fare (pounds sterling)",
        "SurvivedLabel": "Survival status",
    },
    title="Titanic Fare Distribution by Class and Survival Status",
)

box_fig.show()

box_html_path = os.path.join(output_folder, "task3_box_chart.html")
box_fig.write_html(box_html_path)
box_html_path

'c:/Users/PM/Desktop/Internship/AI-Data-Engineering-Internship/Week-6 May_26\\Task3\\task3_box_chart.html'

### Chart 4: Treemap
I chose a treemap for the fourth chart because it is not one of the standard class examples and it works well for a hierarchical part-to-whole summary. The hierarchy shows survival status first, then sex, then class, which makes the composition of passenger groups easy to compare at a glance.

In [10]:
# Build a compact hierarchy for the treemap.
treemap_df = (
    plot_df.groupby(["SurvivedLabel", "Sex", "Pclass"], as_index=False)
    .agg(
        PassengerCount=("PassengerId", "count"),
        MeanAge=("Age", "mean"),
        MeanFare=("Fare", "mean"),
    )
)

treemap_fig = px.treemap(
    treemap_df,
    path=["SurvivedLabel", "Sex", "Pclass"],
    values="PassengerCount",
    color="SurvivedLabel",
    hover_data={
        "PassengerCount": True,
        "MeanAge": ":.1f",
        "MeanFare": ":.2f",
    },
    color_discrete_map={"Did not survive": "#D95F02", "Survived": "#1B9E77"},
    labels={
        "SurvivedLabel": "Survival status",
        "Sex": "Sex",
        "Pclass": "Passenger class",
        "PassengerCount": "Passenger count",
    },
    title="Titanic Passenger Composition Treemap",
)

treemap_fig.update_traces(root_color="lightgrey")
treemap_fig.show()

treemap_html_path = os.path.join(output_folder, "task3_treemap_chart.html")
treemap_fig.write_html(treemap_html_path)
treemap_html_path

'c:/Users/PM/Desktop/Internship/AI-Data-Engineering-Internship/Week-6 May_26\\Task3\\task3_treemap_chart.html'

### Export check
The four HTML files are written to the Task3 folder, one file per chart, so each visual can be opened independently in a browser.

In [11]:
sorted([name for name in os.listdir(output_folder) if name.endswith(".html")])

['task3_bar_chart.html',
 'task3_box_chart.html',
 'task3_scatter_chart.html',
 'task3_treemap_chart.html']